In [ ]:
#| default_exp methods.samplers

In [ ]:
#| export
import numpy as np

def sample_binomial_gap(mu_: float, var_: float, rng = None) -> int:
    """
    Sample a nonnegative integer gap with target mean/variance using a Binomial fit,
    then clamp to ensure the gap length is at least 1 (timeout ≥ 1).

    Logic:
    - Binomial(n,p) has mean μ = n p and variance σ² = n p (1-p) = μ (1-p).
        Given target μ, σ² < μ implies p = 1 - σ²/μ ∈ (0,1) and n ≈ μ / p.
        We round n to an integer ≥ 1 and recompute p = μ / n (clipped into (0,1)).
    - If σ² ≥ μ (outside Binomial under-dispersion), we approximate the Poisson boundary
        by taking a large n with small p that preserves μ, still sampling from Binomial.
    - Finally, we enforce a minimum gap of 1 to guarantee at least one timeout step.
    """
    if rng is None:
        rng = np.random.default_rng()
    if var_ >= mu_:
        # Approximate Poisson boundary: large n, tiny p, maintain mean
        p = 1e-6
        n = max(1, int(round(mu_ / p)))
        p = float(np.clip(mu_ / n, 1e-9, 1 - 1e-9))
    else:
        p_raw = 1.0 - (var_ / mu_)
        p_raw = float(np.clip(p_raw, 1e-9, 1 - 1e-9))
        n = max(1, int(round(mu_ / p_raw)))
        p = float(np.clip(mu_ / n, 1e-9, 1 - 1e-9))

    gap = int(rng.binomial(n, p))
    return max(1, gap)